# Day 13: Decision Trees & Random Forest

## Load Data
This notebook builds on the familiar Iris classification task to demonstrate tree-based models, which work quite differently from the linear/distance-based models used earlier — they learn a series of yes/no decision rules rather than a formula or a distance calculation. `load_iris(return_X_y=True)` returns the feature array `X` and label array `y`, and `feature_names` is fetched separately (from calling `load_iris()` without `return_X_y`) so we have the human-readable column names available later for interpreting feature importance. There's no printed output here, but this cell prepares the train/test data plus `feature_names`, both used throughout the rest of the notebook.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
feature_names = load_iris().feature_names  # human-readable column names, used later for feature importance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Decision Tree
A decision tree classifies data by asking a sequence of simple yes/no questions about feature values (like "is petal length < 2.5?"), splitting the data at each step to make the resulting groups as pure as possible (mostly one class), until it reaches a final decision — this makes trees very easy to interpret and visualize compared to more opaque models. `max_depth=3` limits the tree to at most 3 levels of splits, which prevents it from growing so complex that it just memorizes the training data (overfitting) — a common and important control for tree-based models. The output shows an accuracy of 1.0, meaning this shallow, easy-to-interpret tree still classifies every test sample correctly on this dataset.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=3 caps the tree at 3 levels of splits to avoid overfitting to the training data
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)
print('Accuracy:', tree_model.score(X_test, y_test))

## Random Forest
A single decision tree can be unstable — small changes in the training data can produce a very different tree — so a random forest builds many trees (each trained on a random subset of the data and features) and combines their predictions by majority vote, which usually gives a more accurate and robust model at the cost of being harder to interpret than one tree. `n_estimators=100` sets the forest to build 100 individual trees; `.fit()` trains all of them, and `.score()` evaluates the combined forest's accuracy on the test set exactly like before. The output again shows 1.0 accuracy — on this easy dataset both a single shallow tree and a full forest perform perfectly, but on harder, noisier real-world data the forest typically generalizes better than any single tree.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators=100: build 100 individual decision trees and combine their votes
forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)
print('Accuracy:', forest_model.score(X_test, y_test))

## Feature Importance
One major advantage of tree-based models is that they can tell you which input features actually mattered for the decisions they made, which is valuable for understanding your data, explaining a model to stakeholders, or deciding which features are worth keeping. `forest_model.feature_importances_` returns a score for each feature (summing to 1 across all features) based on how much that feature reduced impurity across all the splits in all the trees — higher means the feature was more useful for distinguishing classes. In the output, `petal length` and `petal width` have by far the highest importance (~0.44 and ~0.42), while `sepal width` barely matters (~0.03) — consistent with what we saw earlier in the EDA, where petal measurements separated species much more clearly than sepal measurements.

In [ ]:
# feature_importances_ scores each feature by how much it reduced impurity across all tree splits
for name, importance in zip(feature_names, forest_model.feature_importances_):
    print(name, ':', round(importance, 3))